In [0]:
from pyspark.sql.functions  import (
    col, 
    sum,
    round,
    lag,
    when,
    lit
)

from pyspark.sql.window import Window

## Read Silver

In [0]:
silver_table = "ev_registration.silver.silver_vehicle_registrations"

gold_monthly_source_df = spark.read.table(silver_table)

print("Silver records:", gold_monthly_source_df.count())

display(gold_monthly_source_df.limit(10))


Silver records: 499995


registration_year,maker,state,rto_code,rto,vehicle_category,vehicle_model,fuel,vehicle_class,sale_type,vehicle_count,ingestion_timestamp,registration_month,registration_month_number,is_ev,ev_type
2023,HYUNDAI MOTOR INDIA LTD,Punjab,PB19,RTO BARNALA,LIGHT MOTOR VEHICLE,CRETA 1.5 CRDI MT EX,DIESEL,Motor Car,Resale,1,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2022,TVS MOTOR COMPANY LTD,Odisha,OR26,NUAPADA RTO,TWO WHEELER(NT),TVS RADEON,PETROL,M-Cycle/Scooter,New,3,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2020,TOYOTA KIRLOSKAR MOTOR PVT LTD,Karnataka,KA63,DHARWAD EAST RTO,LIGHT MOTOR VEHICLE,TOYOTA URBAN CRUISER (HIGH MT),PETROL,Motor Car,New,2,2026-10-06T17:16:18.405Z,Dec,12,0,NON_EV
2022,MARUTI SUZUKI INDIA LTD,Chhattisgarh,CG14,Jashpur DTO,LIGHT MOTOR VEHICLE,SWIFT LXI,PETROL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TATA MOTORS PASSENGER VEHICLES LTD,Karnataka,KA51,ELECTRONIC CITY RTO,LIGHT MOTOR VEHICLE,HARRIER XZ+ 2LBS6PH2 #DRK,DIESEL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2021,HERO ELECTRIC VEHICLES PVT. LTD,Rajasthan,RJ14,JAIPUR (FIRST) RTO,TWO WHEELER(NT),OPTIMA HS 500 ER,ELECTRIC(BOV),M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Jan,1,1,BATTERY_EV
2021,BAJAJ AUTO LTD,Maharashtra,MH40,NAGPUR (RURAL),THREE WHEELER(T),BAJAJ MAXIMA C,DIESEL,Three Wheeler (Goods),Resale,1,2026-10-06T17:16:18.405Z,Jan,1,0,NON_EV
2020,SUZUKI MOTORCYCLE INDIA PVT LTD,Maharashtra,MH28,BULDHANA,TWO WHEELER(NT),ACCESS 125 DISC(ALLOY WHEEL),PETROL,M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV
2023,HONDA MOTORCYCLE AND SCOOTER INDIA (P) LTD,Bihar,BR39,KATIHAR,TWO WHEELER(NT),ACTIVA SMART,PETROL,M-Cycle/Scooter,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TVS MOTOR COMPANY LTD,Arunachal Pradesh,AR01,ITANAGAR CAPITAL REGION,TWO WHEELER(NT),TVS NTORQ 125 XT,PETROL,M-Cycle/Scooter,New,14,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV


## Aggregate Monthly Registrations

In [0]:
gold_monthly_df = (
    gold_monthly_source_df
    .groupBy(
        "registration_year",
        "registration_month",
        "registration_month_number"
    )
    .agg(
        sum("vehicle_count").alias("total_registrations"),
        sum(
            when(col("is_ev") == 1, col("vehicle_count"))
            .otherwise(0)
        ).alias("ev_registrations")
    )
)

display(
    gold_monthly_df
    .orderBy(
        "registration_year",
        "registration_month_number"
    )
)

registration_year,registration_month,registration_month_number,total_registrations,ev_registrations
2020,Apr,4,9957,16
2020,May,5,5232,43
2020,Jun,6,27515,269
2020,Jul,7,30454,214
2020,Aug,8,31895,140
2020,Sep,9,30419,233
2020,Oct,10,33007,190
2020,Nov,11,47238,240
2020,Dec,12,50311,283
2021,Jan,1,41591,292


## Calculate Non-EV Registrations

In [0]:
gold_monthly_df = gold_monthly_df.withColumn(
    "non_ev_registration",
    col("total_registrations") - col("ev_registrations")
)

display(
    gold_monthly_df
    .orderBy(
        "registration_year",
        "registration_month_number"
    )
)

registration_year,registration_month,registration_month_number,total_registrations,ev_registrations,non_ev_registration
2020,Apr,4,9957,16,9941
2020,May,5,5232,43,5189
2020,Jun,6,27515,269,27246
2020,Jul,7,30454,214,30240
2020,Aug,8,31895,140,31755
2020,Sep,9,30419,233,30186
2020,Oct,10,33007,190,32817
2020,Nov,11,47238,240,46998
2020,Dec,12,50311,283,50028
2021,Jan,1,41591,292,41299


## Calculate EV Penetration

In [0]:
gold_monthly_df = gold_monthly_df.withColumn(
    "ev_penetration_pct",
    round(
        when(
            col("total_registrations") > 0,
            (col("ev_registrations") / col("total_registrations")) * 100
        ).otherwise(0),
        2
    )
)

display(
    gold_monthly_df
    .orderBy(
        "registration_year", 
        "registration_month_number"
    )
)

registration_year,registration_month,registration_month_number,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct
2020,Apr,4,9957,16,9941,0.16
2020,May,5,5232,43,5189,0.82
2020,Jun,6,27515,269,27246,0.98
2020,Jul,7,30454,214,30240,0.7
2020,Aug,8,31895,140,31755,0.44
2020,Sep,9,30419,233,30186,0.77
2020,Oct,10,33007,190,32817,0.58
2020,Nov,11,47238,240,46998,0.51
2020,Dec,12,50311,283,50028,0.56
2021,Jan,1,41591,292,41299,0.7


## Create Chronological Window

In [0]:
monthly_window = Window.orderBy(
    "registration_year",
    "registration_month_number"
)

## Month-Over-Month EV Growth

In [0]:
gold_monthly_df = gold_monthly_df.withColumn(
    "previous_month_ev_registrations",
    lag("ev_registrations", 1).over(monthly_window)
)

gold_monthly_df = gold_monthly_df.withColumn(
    "ev_mom_growth_pct",
    round(
        when(
            col("previous_month_ev_registrations") > 0,
            (
                (col("ev_registrations") - col("previous_month_ev_registrations"))
                /
                col("previous_month_ev_registrations")
            ) * 100
        ).otherwise(None),
        2
    )
)

display(
    gold_monthly_df
    .orderBy(
        "registration_year", 
        "registration_month_number"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


registration_year,registration_month,registration_month_number,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct,previous_month_ev_registrations,ev_mom_growth_pct
2020,Apr,4,9957,16,9941,0.16,null,null
2020,May,5,5232,43,5189,0.82,16,168.75
2020,Jun,6,27515,269,27246,0.98,43,525.58
2020,Jul,7,30454,214,30240,0.7,269,-20.45
2020,Aug,8,31895,140,31755,0.44,214,-34.58
2020,Sep,9,30419,233,30186,0.77,140,66.43
2020,Oct,10,33007,190,32817,0.58,233,-18.45
2020,Nov,11,47238,240,46998,0.51,190,26.32
2020,Dec,12,50311,283,50028,0.56,240,17.92
2021,Jan,1,41591,292,41299,0.7,283,3.18


## Year-Over-Year EV Growth

In [0]:
gold_monthly_df = gold_monthly_df.withColumn(
    "previous_year_ev_registrations",
    lag("ev_registrations", 12).over(monthly_window)
)

gold_monthly_df = gold_monthly_df.withColumn(
    "ev_yoy_growth_pct",
    round(
        when(
            col("previous_year_ev_registrations") > 0,
            (
                (
                    col("ev_registrations") - 
                    col("previous_year_ev_registrations")
                )
                / col("previous_year_ev_registrations")
            ) * 100
        ).otherwise(None),
        2
    )
)

display(
    gold_monthly_df
    .orderBy(
        "registration_year",
        "registration_month_number"
    )
)

registration_year,registration_month,registration_month_number,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct,previous_month_ev_registrations,ev_mom_growth_pct,previous_year_ev_registrations,ev_yoy_growth_pct
2020,Apr,4,9957,16,9941,0.16,null,null,null,null
2020,May,5,5232,43,5189,0.82,16,168.75,null,null
2020,Jun,6,27515,269,27246,0.98,43,525.58,null,null
2020,Jul,7,30454,214,30240,0.7,269,-20.45,null,null
2020,Aug,8,31895,140,31755,0.44,214,-34.58,null,null
2020,Sep,9,30419,233,30186,0.77,140,66.43,null,null
2020,Oct,10,33007,190,32817,0.58,233,-18.45,null,null
2020,Nov,11,47238,240,46998,0.51,190,26.32,null,null
2020,Dec,12,50311,283,50028,0.56,240,17.92,null,null
2021,Jan,1,41591,292,41299,0.7,283,3.18,null,null


## Remove Intermediate Columns

In [0]:
gold_monthly_df = gold_monthly_df.drop(
    "previous_month_ev_registrations",
    "previous_year_ev_registrations"
)

display(
    gold_monthly_df
    .orderBy(
        "registration_year",
        "registration_month_number"
    )
)

registration_year,registration_month,registration_month_number,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct,ev_mom_growth_pct,ev_yoy_growth_pct
2020,Apr,4,9957,16,9941,0.16,null,null
2020,May,5,5232,43,5189,0.82,168.75,null
2020,Jun,6,27515,269,27246,0.98,525.58,null
2020,Jul,7,30454,214,30240,0.7,-20.45,null
2020,Aug,8,31895,140,31755,0.44,-34.58,null
2020,Sep,9,30419,233,30186,0.77,66.43,null
2020,Oct,10,33007,190,32817,0.58,-18.45,null
2020,Nov,11,47238,240,46998,0.51,26.32,null
2020,Dec,12,50311,283,50028,0.56,17.92,null
2021,Jan,1,41591,292,41299,0.7,3.18,null


## Final Column Order

In [0]:
gold_monthly_df = gold_monthly_df.select(
    "registration_year",
    "registration_month",
    "registration_month_number",
    "total_registrations",
    "ev_registrations",
    "non_ev_registration",
    "ev_penetration_pct",
    "ev_mom_growth_pct",
    "ev_yoy_growth_pct"
)

gold_monthly_df.printSchema()

display(
    gold_monthly_df
    .orderBy(
        "registration_year",
        "registration_month_number"
    )
)

root
 |-- registration_year: long (nullable = true)
 |-- registration_month: string (nullable = true)
 |-- registration_month_number: integer (nullable = true)
 |-- total_registrations: long (nullable = true)
 |-- ev_registrations: long (nullable = true)
 |-- non_ev_registration: long (nullable = true)
 |-- ev_penetration_pct: double (nullable = true)
 |-- ev_mom_growth_pct: double (nullable = true)
 |-- ev_yoy_growth_pct: double (nullable = true)



registration_year,registration_month,registration_month_number,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct,ev_mom_growth_pct,ev_yoy_growth_pct
2020,Apr,4,9957,16,9941,0.16,null,null
2020,May,5,5232,43,5189,0.82,168.75,null
2020,Jun,6,27515,269,27246,0.98,525.58,null
2020,Jul,7,30454,214,30240,0.7,-20.45,null
2020,Aug,8,31895,140,31755,0.44,-34.58,null
2020,Sep,9,30419,233,30186,0.77,66.43,null
2020,Oct,10,33007,190,32817,0.58,-18.45,null
2020,Nov,11,47238,240,46998,0.51,26.32,null
2020,Dec,12,50311,283,50028,0.56,17.92,null
2021,Jan,1,41591,292,41299,0.7,3.18,null


## Validate Totals

In [0]:
gold_monthly_df.selectExpr(
    "sum(total_registrations) as total_registrations",
    "sum(ev_registrations) as total_ev_registrations",
    "sum(non_ev_registration) as total_non_ev_registration",
).show()

+-------------------+----------------------+-------------------------+
|total_registrations|total_ev_registrations|total_non_ev_registration|
+-------------------+----------------------+-------------------------+
|            2806654|                143873|                  2662781|
+-------------------+----------------------+-------------------------+



## Check Monthly Data Coverage

In [0]:
display(
    gold_monthly_df
    .select(
        "registration_year",
        "registration_month",
        "registration_month_number",
    )
    .orderBy(
        "registration_year",
        "registration_month_number"
    )
)

registration_year,registration_month,registration_month_number
2020,Apr,4
2020,May,5
2020,Jun,6
2020,Jul,7
2020,Aug,8
2020,Sep,9
2020,Oct,10
2020,Nov,11
2020,Dec,12
2021,Jan,1


## Write Gold Delta Table

In [0]:
gold_monthly_table = "ev_registration.gold.gold_ev_monthly"

gold_monthly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(gold_monthly_table)

## Verify The Actual Gold Table

In [0]:
final_gold_monthly_df = spark.read.table(gold_monthly_table)

print("Gold monthly records:",final_gold_monthly_df.count())

final_gold_monthly_df.printSchema()

display(
    final_gold_monthly_df
    .orderBy(
        "registration_year",
        "registration_month_number"
    )
)

Gold monthly records: 69
root
 |-- registration_year: long (nullable = true)
 |-- registration_month: string (nullable = true)
 |-- registration_month_number: integer (nullable = true)
 |-- total_registrations: long (nullable = true)
 |-- ev_registrations: long (nullable = true)
 |-- non_ev_registration: long (nullable = true)
 |-- ev_penetration_pct: double (nullable = true)
 |-- ev_mom_growth_pct: double (nullable = true)
 |-- ev_yoy_growth_pct: double (nullable = true)



registration_year,registration_month,registration_month_number,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct,ev_mom_growth_pct,ev_yoy_growth_pct
2020,Apr,4,9957,16,9941,0.16,null,null
2020,May,5,5232,43,5189,0.82,168.75,null
2020,Jun,6,27515,269,27246,0.98,525.58,null
2020,Jul,7,30454,214,30240,0.7,-20.45,null
2020,Aug,8,31895,140,31755,0.44,-34.58,null
2020,Sep,9,30419,233,30186,0.77,66.43,null
2020,Oct,10,33007,190,32817,0.58,-18.45,null
2020,Nov,11,47238,240,46998,0.51,26.32,null
2020,Dec,12,50311,283,50028,0.56,17.92,null
2021,Jan,1,41591,292,41299,0.7,3.18,null
